# 06 — Baselines: B0 cycle recon + (future) B1 fix + multi-seed

**Phase 1 의 architectural superiority 의 direct evidence path**.

v6_long 의 plateau 도달 (recon 0.1205) = Stage 1 architecture 의 capacity ceiling. 이게 B0 (no MoE, capacity-matched MLP) 보다 *낮은지* 확인.

## Setup

- B0 trainable ~50M (Phase 1 의 62M 와 capacity matched)
- 같은 fact_emb cache + 같은 corpus split → fair 비교
- B0 cycle: fact_emb → CapacityMLP(width=4500, n_hidden=3) → sub_kg → FactDecoder(d_bottleneck=64) → recon

## Verdict criterion

- B0 recon < 0.87 → **Phase 1 의 MoE-KG-cycle architecture 의 architectural superiority direct evidence**
- B0 recon ≈ 0.88 → architecture 차이 무관, recon 측면 parity. Paradigm value = *interpretable structure* 측면만

**Pre-req**: `b0_v0` 학습 완료 (03_baselines.ipynb).

In [ ]:
import os, sys
BASE = '/content/drive/MyDrive/sideproject'
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir(BASE)
if BASE not in sys.path:
    sys.path.insert(0, BASE)
print('cwd =', os.getcwd())

## A. B0 cycle recon eval

In [ ]:
import json
import torch
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader

from phase1.baselines.generic_baseline import GenericBaseline
from phase1.data import CorpusConfig, Phase1Dataset, load_phase1_corpus
from phase1.train import CONFIG_NAME, MODEL_NAME

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
run_id = 'b0_v0'
run_dir = Path(f'out/phase1/{run_id}')
cfg = json.loads((run_dir / CONFIG_NAME).read_text())

ccfg = CorpusConfig(encoder_name=cfg['encoder_name'], encoder_max_length=cfg['encoder_max_length'])
corpus, fact_emb = load_phase1_corpus(cfg=ccfg)

model = GenericBaseline(
    encoder_name=cfg['encoder_name'],
    d_bottleneck=cfg.get('d_bottleneck', 64),
    mlp_width=cfg.get('mlp_width', 4500),
    mlp_n_hidden=cfg.get('mlp_n_hidden', 3),
).to(device)
model.load_state_dict(torch.load(run_dir / MODEL_NAME, map_location=device), strict=False)
model.eval()

test_mask = (corpus['split'] == 'test').values
test_loader = DataLoader(
    Phase1Dataset(fact_emb[test_mask], corpus['user_id'].values[test_mask]),
    batch_size=128, shuffle=False,
)

total_cos = 0.0
total_mse = 0.0
n = 0
with torch.no_grad():
    for fact, _ in test_loader:
        fact = fact.to(device)
        recon = model(fact)['recon']
        total_cos += F.cosine_similarity(fact, recon, dim=-1).sum().item()
        total_mse += ((fact - recon) ** 2).mean(dim=-1).sum().item()
        n += fact.size(0)

mean_cosine = total_cos / n
mean_mse = total_mse / n
print(f'[b0 recon] mean_cosine={mean_cosine:.4f}  mse={mean_mse:.6f}')

out_json = run_dir / 'eval.json'
payload = {'cycle_reconstruction': {'mean_cosine': mean_cosine, 'mean_mse': mean_mse}}
if out_json.exists():
    existing = json.loads(out_json.read_text())
    existing.update(payload)
    payload = existing
out_json.write_text(json.dumps(payload, indent=2))
print(f'[b0 recon] saved → {out_json}')

## B. 5-way 비교 — Phase 1 vs B0

In [ ]:
from phase1.cluster_analysis import compare_runs

compare_runs(['ph1_v3_minimal', 'ph1_v4_diverse', 'ph1_v5_arch', 'ph1_v6_long', 'b0_v0'])